In [32]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pypsa

sys.path.append(str(Path.cwd().parent / 'scripts'))
from _tyndp_helpers import _extract_scenario_values, _extract_scenario_values_rowwise

In [33]:
year = 2035
scenario = 'GA'

In [34]:
valpath = Path.cwd().parent / 'data' / 'TYNDP_2024-Scenario-Report-Data-Figures_240522.xlsx'

In [35]:
scenario_mapper = {
    'NT': 'National Trends',
    'DE': 'Distributed Energy',
    'GA': 'Global Ambition',
    'REF': 'Reference',
}

In [36]:
# prenetwork
# n = pypsa.Network(Path.cwd().parent / 'resources' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_{}.nc'.format(year))
n = pypsa.Network(Path.cwd().parent / 'results' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_{}.nc'.format(year))

INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


In [37]:
eu27_countries = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE", "GR", "HU",
    "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT", "RO", "SK", "SI", "ES", "SE",
]
eu27_buses = n.buses.index[
    (n.buses.country.isin(eu27_countries)) &
    (n.buses.carrier == 'AC')
]
all_eu27_buses = n.buses.index[
    (n.buses.country.isin(eu27_countries))
]

In [38]:
# adjust EV demand
def final_electricity_transport(path):
    return _extract_scenario_values(path, sheet_name='6-', row_label='Transport')
ev_demand = final_electricity_transport(valpath)

In [39]:
ev_demand

{'National Trends': {2030: 247.64143184472053, 2040: 488.45309616768986},
 'Distributed Energy': {2040: 616.8067207371424, 2050: 849.9178634898052},
 'Global Ambition': {2040: 533.8023438415136, 2050: 779.9316227533446},
 'Reference': {2019: 50.09668955008888}}

In [40]:
weight = (year - 2030) / (2040 - 2030)

tyndp_p_set = ev_demand['National Trends'][2030] * (1 - weight) + ev_demand[scenario_mapper[scenario]][2040] * weight

In [41]:
n.loads.carrier.unique()

array(['electricity', 'land transport EV', 'land transport oil',
       'urban central heat', 'solid biomass for industry',
       'gas for industry', 'H2 for industry', 'industry methanol',
       'naphtha for industry', 'low-temperature heat for industry',
       'industry electricity', 'process emissions', 'NH3',
       'coal for industry', 'shipping methanol', 'shipping oil',
       'kerosene for aviation', 'agriculture electricity',
       'agriculture heat', 'agriculture machinery electric',
       'agriculture machinery oil', 'rural heat', 'urban decentral heat'],
      dtype=object)

In [ ]:
tyndp_p_set

390.72188784311703

In [56]:
idx = pd.IndexSlice
eb = n.statistics.energy_balance().loc[idx[:,:,'urban central heat']].sort_values()
eb = eb.loc[eb > 0]

In [93]:
def to_ac_bus(x):
    return ' '.join(x.split(' ')[:2])

In [ ]:
uc = pd.Index(n.loads.loc[n.loads.carrier == 'urban central heat', 'bus'])
lt = pd.Index(n.loads.loc[n.loads.carrier == 'low-temperature heat for industry', 'bus'])
inter = uc.intersection(lt)
inter = inter.map(to_ac_bus)

Index(['AL0 0', 'AT0 0', 'BA0 0', 'BE0 0', 'BG0 0', 'CH0 0', 'CZ0 0', 'DE0 0',
       'DE0 1', 'DE0 2', 'DE0 3', 'DE0 4', 'DE0 5', 'DK0 0', 'DK1 0', 'EE0 0',
       'ES0 0', 'ES6 0', 'FI1 0', 'FR0 0', 'FR0 1', 'FR0 2', 'FR0 3', 'FR0 4',
       'GB2 0', 'GB2 1', 'GB3 0', 'GR0 0', 'HR0 0', 'HU0 0', 'IE3 0', 'IT0 0',
       'IT0 1', 'IT4 0', 'LT0 0', 'LU0 0', 'LV0 0', 'ME0 0', 'MK0 0', 'NL0 0',
       'NO1 0', 'PL0 0', 'PT0 0', 'RO0 0', 'RS0 0', 'SE1 0', 'SI0 0', 'SK0 0',
       'XK0 0'],
      dtype='object', name='bus')

In [101]:
urban_central_load = (
    n.loads_t
    .p_set[inter + ' urban central heat']
    .add(
        n.loads.loc[
            inter + ' low-temperature heat for industry',
            'p_set'
        ].to_list(), axis=1
    )
)
urban_central_load.head()

Load,AL0 0 urban central heat,AT0 0 urban central heat,BA0 0 urban central heat,BE0 0 urban central heat,BG0 0 urban central heat,CH0 0 urban central heat,CZ0 0 urban central heat,DE0 0 urban central heat,DE0 1 urban central heat,DE0 2 urban central heat,...,NL0 0 urban central heat,NO1 0 urban central heat,PL0 0 urban central heat,PT0 0 urban central heat,RO0 0 urban central heat,RS0 0 urban central heat,SE1 0 urban central heat,SI0 0 urban central heat,SK0 0 urban central heat,XK0 0 urban central heat
snapshot,,,,,,,,,,,,,,,,,,,,,
2024-01-01,102.337371,3317.906403,379.109535,4157.538804,948.170613,2361.599666,4326.822886,5804.102535,1514.458559,9504.194662,...,6975.355667,3967.655878,13692.505717,1582.635372,2330.242924,1380.607329,14305.332059,460.425930,1146.491057,17.866889
2024-01-08,171.047584,4644.255315,792.569289,7499.436565,1721.044785,3132.157962,6897.555770,7877.550211,2527.751234,15394.269623,...,10607.119158,2881.746396,18030.123377,1748.428167,4038.059122,2786.752243,8870.904140,722.988408,1845.034107,17.866889
2024-01-15,119.040118,4282.974369,566.294687,7278.633580,1271.146044,2923.761329,6039.388243,7285.296659,2347.365350,14817.961319,...,10082.874512,3865.342348,15487.822543,1130.936083,3216.142369,2004.417017,13750.665034,644.508180,1579.330151,17.866889
2024-01-22,168.547979,3219.838992,600.722585,3931.593090,1447.149128,2025.414406,4398.389930,4690.779354,1489.587958,8792.749758,...,6315.727386,2508.914039,11320.186377,925.219575,3292.066923,2020.011380,8575.360141,585.801348,1347.813150,17.866889
2024-01-29,161.588004,3117.257409,562.775276,3696.822848,1341.066956,2097.294476,4380.848669,4544.884792,1426.628666,8353.564635,...,5981.738674,2484.706764,10709.750599,1046.048523,3107.197286,1935.443155,7773.970285,561.441294,1306.838639,17.866889


In [102]:
p_max_pu = urban_central_load.div(urban_central_load.max(), axis=1)
p_max_pu.head()

Load,AL0 0 urban central heat,AT0 0 urban central heat,BA0 0 urban central heat,BE0 0 urban central heat,BG0 0 urban central heat,CH0 0 urban central heat,CZ0 0 urban central heat,DE0 0 urban central heat,DE0 1 urban central heat,DE0 2 urban central heat,...,NL0 0 urban central heat,NO1 0 urban central heat,PL0 0 urban central heat,PT0 0 urban central heat,RO0 0 urban central heat,RS0 0 urban central heat,SE1 0 urban central heat,SI0 0 urban central heat,SK0 0 urban central heat,XK0 0 urban central heat
snapshot,,,,,,,,,,,,,,,,,,,,,
2024-01-01,0.598298,0.714411,0.478330,0.554380,0.550927,0.753985,0.627298,0.736790,0.599133,0.617385,...,0.657611,1.000000,0.759424,0.726365,0.577070,0.495418,1.000000,0.636837,0.621393,1.0
2024-01-08,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,0.726310,1.000000,0.802457,1.000000,1.000000,0.620112,1.000000,1.000000,1.0
2024-01-15,0.695947,0.922209,0.714505,0.970557,0.738590,0.933465,0.875584,0.924818,0.928638,0.962563,...,0.950576,0.974213,0.858997,0.519054,0.796457,0.719266,0.961227,0.891450,0.855990,1.0
2024-01-22,0.985386,0.693295,0.757943,0.524252,0.840855,0.646651,0.637674,0.595462,0.589294,0.571170,...,0.595423,0.632342,0.627849,0.424638,0.815260,0.724862,0.599452,0.810250,0.730509,1.0
2024-01-29,0.944696,0.671207,0.710064,0.492947,0.779217,0.669600,0.635131,0.576941,0.564386,0.542641,...,0.563936,0.626240,0.593992,0.480094,0.769478,0.694516,0.543432,0.776556,0.708301,1.0


In [ ]:
urban_central_load.head()

Load,AL0 0 urban central heat,AT0 0 urban central heat,BA0 0 urban central heat,BE0 0 urban central heat,BG0 0 urban central heat,CH0 0 urban central heat,CZ0 0 urban central heat,DE0 0 urban central heat,DE0 1 urban central heat,DE0 2 urban central heat,...,NL0 0 urban central heat,NO1 0 urban central heat,PL0 0 urban central heat,PT0 0 urban central heat,RO0 0 urban central heat,RS0 0 urban central heat,SE1 0 urban central heat,SI0 0 urban central heat,SK0 0 urban central heat,XK0 0 urban central heat
snapshot,,,,,,,,,,,,,,,,,,,,,
2024-01-01,102.337371,3317.906403,379.109535,4157.538804,948.170613,2361.599666,4326.822886,5804.102535,1514.458559,9504.194662,...,6975.355667,3967.655878,13692.505717,1582.635372,2330.242924,1380.607329,14305.332059,460.425930,1146.491057,17.866889
2024-01-08,171.047584,4644.255315,792.569289,7499.436565,1721.044785,3132.157962,6897.555770,7877.550211,2527.751234,15394.269623,...,10607.119158,2881.746396,18030.123377,1748.428167,4038.059122,2786.752243,8870.904140,722.988408,1845.034107,17.866889
2024-01-15,119.040118,4282.974369,566.294687,7278.633580,1271.146044,2923.761329,6039.388243,7285.296659,2347.365350,14817.961319,...,10082.874512,3865.342348,15487.822543,1130.936083,3216.142369,2004.417017,13750.665034,644.508180,1579.330151,17.866889
2024-01-22,168.547979,3219.838992,600.722585,3931.593090,1447.149128,2025.414406,4398.389930,4690.779354,1489.587958,8792.749758,...,6315.727386,2508.914039,11320.186377,925.219575,3292.066923,2020.011380,8575.360141,585.801348,1347.813150,17.866889
2024-01-29,161.588004,3117.257409,562.775276,3696.822848,1341.066956,2097.294476,4380.848669,4544.884792,1426.628666,8353.564635,...,5981.738674,2484.706764,10709.750599,1046.048523,3107.197286,1935.443155,7773.970285,561.441294,1306.838639,17.866889


In [155]:
eff = 1.035
w = n.snapshot_weightings['generators'].iloc[0]
target_generation = 200_000_000

# cap * p_max_pu.mean() * w * len(n.shapshots) / eff
cap = target_generation * eff / (p_max_pu.mean() * w * len(n.snapshots))
# cap * p_max_pu

In [ ]:
existing = pd.read_csv(Path.cwd().parent / 'resources' / 'existing_heating_distribution_base_s_50_2030.csv', header=[0, 1], index_col=0)
idx = pd.IndexSlice

res = existing.loc[:, idx['residential urban decentral', :]]
res.columns = res.columns.get_level_values(1)
ser = existing.loc[:, idx['services urban decentral', :]]
ser.columns = ser.columns.get_level_values(1)

existing = (res + ser)['gas boiler']

In [156]:
target_distribution = target_generation * existing / existing.sum()

In [1]:
n.links.loc[n.links.carrier == 'urban central gas boiler']

NameError: name 'n' is not defined

In [ ]:
capacities = pd.Series(index=existing.index, name='capacity')

for bus in existing.index:

    current_trajectory = (existing.loc[bus] * p_max_pu[bus + ' urban central heat'] / eff).sum() * w

    factor = target_distribution.loc[bus] / current_trajectory
    print(np.round(factor, 3))

    p_set = p_max_pu[bus + ' urban central heat'] * factor * existing.loc[bus]
    p_nom = p_set.max()

    # print('target', target_distribution.loc[bus])
    # print(p_set.sum() * w / eff)

AL0 0
nan
AT0 0
0.089
BA0 0
nan
BE0 0
0.113
BG0 0
0.104
CH0 0
nan
CZ0 0
0.101
DE0 0
0.098
DE0 1
0.103
DE0 2
0.1
DE0 3
0.102
DE0 4
0.105
DE0 5
0.097
DK0 0
0.095
DK1 0
0.094
EE0 0
0.133
ES0 0
0.102
ES6 0
0.168
FI1 0
nan
FR0 0
0.125
FR0 1
0.098
FR0 2
0.118
FR0 3
0.133
FR0 4
0.106
FR5 0


/tmp/ipykernel_656236/1324279592.py:8: RuntimeWarning: invalid value encountered in scalar divide
  factor = target_distribution.loc[bus] / current_trajectory


KeyError: 'FR5 0 urban central heat'

Load,AL0 0 urban central heat,AT0 0 urban central heat,BA0 0 urban central heat,BE0 0 urban central heat,BG0 0 urban central heat,CH0 0 urban central heat,CZ0 0 urban central heat,DE0 0 urban central heat,DE0 1 urban central heat,DE0 2 urban central heat,...,NL0 0 urban central heat,NO1 0 urban central heat,PL0 0 urban central heat,PT0 0 urban central heat,RO0 0 urban central heat,RS0 0 urban central heat,SE1 0 urban central heat,SI0 0 urban central heat,SK0 0 urban central heat,XK0 0 urban central heat
snapshot,,,,,,,,,,,,,,,,,,,,,
2024-01-01,3244.141031,3771.884913,3389.833638,3711.345442,3392.685925,3526.469510,3742.236311,4268.580378,3633.459581,3660.277561,...,3815.233007,4818.359507,4777.753360,4854.801606,3550.060686,3594.395227,5724.287285,3596.784126,3771.296192,2324.797844
2024-01-08,5422.285906,5279.713900,7086.812094,6694.585674,6158.136879,4677.109211,5965.643694,5793.480741,6064.531702,5928.676934,...,5801.658446,3499.620575,6291.286951,5363.378085,6151.871458,7255.277263,3549.697663,5647.886145,6069.101070,2324.797844
2024-01-15,3773.625669,4869.000039,5063.562377,6497.479600,4548.336801,4365.919985,5223.421105,5357.912645,5631.753391,5706.727739,...,5514.918160,4694.108972,5404.196845,3469.194743,4899.704004,5218.476543,5502.336940,5034.809376,5195.087871,2324.797844
2024-01-22,5343.047303,3660.399253,5371.401770,3509.648565,5178.100237,3024.459331,3804.134105,3449.795828,3573.790519,3386.284244,...,3454.443434,3046.849370,3949.975236,2838.150568,5015.372963,5259.076286,3431.435553,4576.199670,4433.530092,2324.797844
2024-01-29,5122.412940,3543.781760,5032.093326,3300.074220,4798.523519,3131.794575,3788.962802,3342.498849,3422.739807,3217.144248,...,3271.765329,3017.451822,3736.974661,3208.798525,4733.729180,5038.903890,3110.758917,4385.902273,4298.747515,2324.797844
2024-02-05,3100.254432,2779.966608,2654.420714,2920.363569,2275.429130,2973.326976,2831.457463,3562.493736,2954.837837,2988.867508,...,3213.406145,4547.382535,3631.533973,3507.237344,2383.165645,2393.741043,4957.028486,2763.230403,2690.836694,2324.797844
2024-02-12,3884.064117,3048.723587,3701.118960,2714.857945,3639.262400,2778.715702,3047.361831,2869.670435,2776.064935,2710.921559,...,2744.512627,3683.686216,3287.494738,2131.664544,3561.232391,3814.033039,4104.052446,3286.606590,3475.479674,2324.797844
2024-02-19,3237.695638,3280.345740,3284.581738,3475.818712,3083.362037,3241.968550,3134.106017,3200.342965,3203.183535,3250.845375,...,3328.586948,2966.790790,3133.386872,3200.617516,3162.035763,3004.766805,3249.047350,3165.660035,3159.657622,2324.797844
2024-02-26,2688.283590,2980.260780,2465.215827,3758.603259,2937.233814,3049.721263,2910.991463,3613.313185,3375.297659,3357.332028,...,3504.693382,2776.768850,2649.949621,5023.270205,2678.998677,2138.338702,2962.322699,2948.665262,2340.909290,2324.797844


In [ ]:
(cap * eff * p_max_pu * w).sum().sum() = 

In [ ]:
energy_shares = eb / eb.sum()
energy_shares.round(2)

component  carrier                           
Link       Sabatier                              0.00
           Fischer-Tropsch                       0.00
           urban central gas CHP CC              0.00
           urban central solid biomass CHP CC    0.00
Generator  urban central solar thermal           0.00
Link       H2 Electrolysis                       0.00
           methanolisation                       0.01
           Haber-Bosch                           0.02
           urban central gas CHP                 0.07
           urban central gas boiler              0.25
           urban central solid biomass CHP       0.28
           urban central air heat pump           0.37
dtype: float64

In [ ]:
gas_boiler_target = 200 # TWh

In [ ]:

capacity_factor = n.loads

In [55]:
n.links.loc[n.links.carrier == 'urban central gas boiler']

,bus0,bus1,type,carrier,efficiency,active,build_year,lifetime,p_nom,p_nom_mod,...,tags,location,dc,project_status,under_construction,geometry,reversed,underground,length_original,voltage
Link,,,,,,,,,,,,,,,,,,,,,
AL0 0 urban central gas boiler,AL0 0 gas,AL0 0 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN
AT0 0 urban central gas boiler,AT0 0 gas,AT0 0 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN
BA0 0 urban central gas boiler,BA0 0 gas,BA0 0 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN
BE0 0 urban central gas boiler,BE0 0 gas,BE0 0 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN
BG0 0 urban central gas boiler,BG0 0 gas,BG0 0 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN
CH0 0 urban central gas boiler,CH0 0 gas,CH0 0 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN
CZ0 0 urban central gas boiler,CZ0 0 gas,CZ0 0 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN
DE0 0 urban central gas boiler,DE0 0 gas,DE0 0 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN
DE0 1 urban central gas boiler,DE0 1 gas,DE0 1 urban central heat,,urban central gas boiler,1.035,True,0,25.0,0.0,0.0,...,,,NaN,,NaN,,False,NaN,0.0,NaN


In [22]:
n = pypsa.Network(Path.cwd().parent / 'resources' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_{}.nc'.format(year))

loads = n.loads.index[
    (n.loads.carrier == 'land transport EV') &
    (n.loads.index.str.contains('|'.join(eu27_countries)))
]
w = n.snapshot_weightings['generators'].iloc[0]

print('before')
print(n.loads_t.p_set[loads].sum().sum() * w * 1e-6)

factor = tyndp_p_set * 1e6 / (n.loads_t.p_set[loads].sum().sum() * w)
n.loads_t.p_set[loads] *= factor

print('after')
print(n.loads_t.p_set[loads].sum().sum() * w * 1e-6)

INFO:pypsa.io:Imported network base_s_50__168H-T-H-B-I-A-dist1_2035.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


before
380.99519180496895
after
390.7218878431169


In [23]:
eudh_buses = n.buses.index[
    (n.buses.carrier == 'urban central heat') &
    (n.buses.index.str.contains('|'.join(eu27_countries)))
]

In [14]:
eudh_buses

Index(['AT0 0 urban central heat', 'BE0 0 urban central heat',
       'BG0 0 urban central heat', 'CZ0 0 urban central heat',
       'DE0 0 urban central heat', 'DE0 1 urban central heat',
       'DE0 2 urban central heat', 'DE0 3 urban central heat',
       'DE0 4 urban central heat', 'DE0 5 urban central heat',
       'DK0 0 urban central heat', 'DK1 0 urban central heat',
       'EE0 0 urban central heat', 'ES0 0 urban central heat',
       'ES6 0 urban central heat', 'FI1 0 urban central heat',
       'FR0 0 urban central heat', 'FR0 1 urban central heat',
       'FR0 2 urban central heat', 'FR0 3 urban central heat',
       'FR0 4 urban central heat', 'GR0 0 urban central heat',
       'HR0 0 urban central heat', 'HU0 0 urban central heat',
       'IE3 0 urban central heat', 'IT0 0 urban central heat',
       'IT0 1 urban central heat', 'IT4 0 urban central heat',
       'LT0 0 urban central heat', 'LU0 0 urban central heat',
       'LV0 0 urban central heat', 'NL0 0 urban central

In [ ]:
dh_links = n.links.index[
    (n.links.bus0.isin(eudh_buses)) |
    (n.links.bus1.isin(eudh_buses)) 
]

In [ ]:
n = pypsa.Network(Path.cwd().parent / 'resources' / 'networks' / 'base_s_50__168H-T-H-B-I-A-dist1_{}.nc'.format(year))